In [1]:
import openai, time

In [ ]:
# api_key = os.getenv('OPENAI_API_KEY')
api_key = 'sk-....................'
client = openai.OpenAI(api_key=api_key)

In [ ]:
# 1. Criar o Assistente com a ferramenta file_search
system_prompt = "Voce é um assistente especializado em análise de documentos. Responda apenas com base no conteúdo dos arquivos fornecidos."
model = "gpt-4o-mini"
assistant = client.beta.assistants.create(
    name=f"Assistente de Análise de Documentos ({time.strftime('%Y-%m-%d %H:%M')})",
    instructions=system_prompt,
    model=model,
    tools=[{"type": "file_search"}],
)
print(f"Assistente criado com ID: {assistant.id}")

In [3]:
# 2. Criar um Vector Store
vector_store = client.vector_stores.create(
    name=f"Documentos da Sessão ({time.strftime('%Y-%m-%d %H:%M')})",
)
print(f"Vector Store criado com ID: {vector_store.id}")

Vector Store criado com ID: vs_68c089c193188191a864761b3107349b


In [5]:
file_paths = [r"C:\Users\edson.eab\Downloads\2021.0017256 - Até pág. 126.pdf"]
# file_paths = [r"C:\Users\edson.eab\Downloads\teste_ff.txt"]

# Upload files:
for path in file_paths:
    client.vector_stores.files.upload_and_poll(        
        vector_store_id=vector_store.id,
        file=open(path, "rb")
    )

In [ ]:
user_query = "O que está descrito na fase 2?"

results = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query=user_query,
)
print(results.data)

[]

In [6]:
client.vector_stores.files.list(
    vector_store_id="vs_68c089c193188191a864761b3107349b"
)

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-Qfmr13KU7Q1qiJqgs331av', created_at=1757448660, last_error=None, object='vector_store.file', status='completed', usage_bytes=14806, vector_store_id='vs_68c089c193188191a864761b3107349b', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static')), VectorStoreFile(id='file-UvxtMCr5cmkbiR3AK43HKA', created_at=1757448649, last_error=None, object='vector_store.file', status='completed', usage_bytes=1122, vector_store_id='vs_68c089c193188191a864761b3107349b', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-Qfmr13KU7Q1qiJqgs331av', last_id='file-UvxtMCr5cmkbiR3AK43HKA')

In [ ]:
client.vector_stores.files.retrieve(
    vector_store_id="vs_68bf822ed72081919dadc2082eff3430",
    file_id="file-MEK1iyx62B5pzunN4fwykn"
)

VectorStoreFile(id='file-MEK1iyx62B5pzunN4fwykn', created_at=1757381232, last_error=None, object='vector_store.file', status='completed', usage_bytes=14806, vector_store_id='vs_68bf822ed72081919dadc2082eff3430', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [29]:
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open(r"C:\Users\edson.eab\Downloads\2021.0017256 - Até pág. 126.pdf", "rb")]
)
print("Status:", file_batch.status)
print("Contagens:", file_batch.file_counts)


Status: completed
Contagens: FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [64]:
files = client.vector_stores.files.list(vector_store.id)
for f in files.data:
    print(f.__dict__)

{
    'id': 'file-MLgJDHt4T8XyjKozKRjdUF',
    'created_at': 1757386873,
    'last_error': None,
    'object': 'vector_store.file',
    'status': 'completed',
    'usage_bytes': 1122,
    'vector_store_id': 'vs_68bf985d8de08191846fd8c00dff95ef',
    'attributes': {},
    'chunking_strategy': StaticFileChunkingStrategyObject(
        static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800),
        type='static'
    )
}

In [ ]:
from rich import print
response = client.responses.create(
    model="gpt-4o-mini",
    input="Qual o nome do delegado de polícia envolvido no caso anexo?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["vs_68bf822ed72081919dadc2082eff3430"]
    }],
    include=["file_search_call.results"]
)
print(response)

In [65]:
import requests

headers = {
    "Authorization": f"Bearer {api_key}"
}
url = "https://api.openai.com/v1/vector_stores/vs_68bf985d8de08191846fd8c00dff95ef/files/file-MLgJDHt4T8XyjKozKRjdUF/content"

response = requests.get(url, headers=headers)
print(response.json())

{
    'object': 'vector_store.file_content.page',
    'data': [
        {
            'type': 'text',
            'text': 'Fase 1 Introdução ao projeto.\r\nFase 2 Implementação dos módulos.\r\nFase 3 Validação dos 
resultados.'
        }
    ],
    'has_more': False,
    'next_page': None
}

In [ ]:
file_response = client.files.create(file=open(r"C:\Users\edson.eab\Downloads\teste_ff.txt", "rb"), purpose="assistants")

client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_response.id
)

while vector_store.file_counts.in_progress > 0:
    time.sleep(1)
    vector_store = client.beta.vector_stores.retrieve(vector_store.id)

files = client.vector_stores.files.list(vector_store.id)
for f in files.data:
    print(f.__dict__)
    

In [ ]:
assistant = client.beta.assistants.create(
    name="Leitor de arquivos",
    instructions="Você deve responder exclusivamente com base nos documentos vinculados. Se não encontrar a resposta nos arquivos, diga: 'Não encontrado no documento'.",
    model="gpt-4o",
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
    tools=[{"type": "file_search"}]
)


In [58]:
thread = client.beta.threads.create()

message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="Quais são os principais tópicos abordados no documento?"
)

In [59]:
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id
)

# Aguardar a conclusão
import time
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run_status.status == "completed":
        break
    time.sleep(1)

# Obter resposta
messages = client.beta.threads.messages.list(thread_id=thread.id)
print(messages.data[0].content[0].text.value)


Não consegui localizar automaticamente os principais tópicos do documento. Poderia dar uma olhada para identificar 
os conteúdos principais para que eu possa ajudar você mais efetivamente? Ou, se preferir, você pode me fornecer um 
resumo ou os cabeçalhos principais para que eu possa oferecer uma assistência direcionada.

In [70]:
print(f"vector_store.id: {vector_store.id}\n")

# Crie a thread vinculada ao vector_store
thread = client.beta.threads.create(
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
)

# Adicione a pergunta
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="O que está descrito na fase 2?"
)

# Execute o run
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=assistant.id
)

# Recupere a resposta
messages = client.beta.threads.messages.list(thread_id=thread.id, order="desc", limit=5)
for m in messages.data:
    if m.role == "assistant":
        for c in m.content:
            if c.type == "text":
                print(c.text.value)



vector_store.id: vs_68bf985d8de08191846fd8c00dff95ef

Parece que não encontrei informações específicas sobre "fase 2" nos documentos fornecidos. Poderia fornecer mais 
detalhes ou contexto sobre a "fase 2" que está procurando? Isso pode me ajudar a realizar uma pesquisa mais eficaz.

In [8]:
system_prompt = "Voce é um assistente especializado em análise de documentos. Responda apenas com base no conteúdo dos arquivos fornecidos. \n"
query_input = "Qual o nome do delegado de polícia envolvido no caso anexo?"

response = client.responses.create(
    model="gpt-4.1-mini",
    tools=[{
      "type": "file_search",
      "vector_store_ids": ["vs_68c089c193188191a864761b3107349b"],
      "max_num_results": 20
    }],
    input= system_prompt + query_input,
)

print(response)


Response(id='resp_68c08a9e76ec819681a09042716ec1340ae80d279e59adc3', created_at=1757448862.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseFileSearchToolCall(id='fs_68c08a9f51308196ace0c2479d93de120ae80d279e59adc3', queries=['nome do delegado de polícia envolvido no caso'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_68c08aa17da88196be70fc05610013440ae80d279e59adc3', content=[ResponseOutputText(annotations=[], text='Não foi possível encontrar o nome do delegado de polícia envolvido no caso com base no conteúdo dos arquivos fornecidos. Se puder indicar uma parte específica do documento para que eu faça uma busca mais direcionada, por favor me avise.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FileSearchTool(type='file_search', vector_